# Deploy M-Optimus Model Package from AWS Marketplace

M-Optimus is a multimodal foundation model for histology, developed by Bioptimus.

The model integrates whole slide histology images with bulk RNA-seq data to produce rich multimodal representations. For more information, please refer to this [page](https://www.bioptimus.com/m-optimus).

M-Optimus can extract powerful features from histology images (with optional RNA-seq context) for various downstream applications, such as mutation prediction, survival analysis, or tissue classification. The package also includes a dedicated tissue segmentation model.

This sample notebook shows you how to deploy M-Optimus using Amazon SageMaker.

> **Note**: This is a reference notebook and it cannot run unless you make changes suggested in the notebook.

## Pre-requisites:
1. **Note**: This notebook contains elements which render correctly in Jupyter interface. Open this notebook from an Amazon SageMaker Notebook Instance or Amazon SageMaker Studio.
1. Ensure that IAM role used has **AmazonSageMakerFullAccess**
1. To deploy this ML model successfully, ensure that:
    1. Either your IAM role has these three permissions and you have authority to make AWS Marketplace subscriptions in the AWS account used: 
        1. **aws-marketplace:ViewSubscriptions**
        1. **aws-marketplace:Unsubscribe**
        1. **aws-marketplace:Subscribe**  
    2. or your AWS account has a subscription to M-Optimus on AWS Marketplace. If so, skip step: [Subscribe to the model package](#1.-Subscribe-to-the-model-package).

## Contents:
1. [Subscribe to the model package](#1.-Subscribe-to-the-model-package)
2. [Create an endpoint and perform real-time inference](#2.-Create-an-endpoint-and-perform-real-time-inference)
   1. [Create an endpoint](#A.-Create-an-endpoint)
   2. [Load input payload (embedding mode)](#B.-Load-input-payload-(embedding-mode))
   3. [Perform real-time inference (embedding mode)](#C.-Perform-real-time-inference-(embedding-mode))
   4. [Perform real-time inference (prediction mode)](#D.-Perform-real-time-inference-(prediction-mode))
   5. [Test tissue segmentation model](#E.-Test-tissue-segmentation-model)
   6. [Delete the endpoint](#F.-Delete-the-endpoint-and-model)
3. [Perform batch inference](#3.-Perform-batch-inference)
4. [Clean-up](#4.-Clean-up)
    1. [Delete the model](#A.-Delete-the-model)
    2. [Unsubscribe to the listing (optional)](#B.-Unsubscribe-to-the-listing-(optional))

## Usage instructions
You can run this notebook one cell at a time (By using Shift+Enter for running a cell).

## 1. Subscribe to the model package

To subscribe to the model package:
1. Open the model package listing page for M-Optimus on AWS Marketplace.
1. On the AWS Marketplace listing, click on the **Continue to subscribe** button.
1. On the **Subscribe to this software** page, review and click on **"Accept Offer"** if you and your organization agrees with EULA, pricing, and support terms. 
1. Once you click on **Continue to configuration button** and then choose a **region**, you will see a **Product Arn** displayed. This is the model package ARN that you need to specify while creating a deployable model using Boto3. Copy the ARN corresponding to your region and specify the same in the following cell.

In [ ]:
# Please replace the ARN below with your own after subscribing to the M-Optimus model package
# on AWS Marketplace. To find your ARN:
#   1. Go to AWS Marketplace and subscribe to M-Optimus
#   2. Click "Continue to configuration"
#   3. Select your AWS region
#   4. Copy the "Product Arn" displayed on that page
model_package_arn = "<REPLACE WITH YOUR MODEL PACKAGE ARN HERE>"

In [ ]:
# The code was executed with python 3.13.7
%pip install sagemaker=="2.254.1"
%pip install boto3=="1.42.2"

In [ ]:
import json
import os
import time
from datetime import datetime
from sagemaker import ModelPackage
import sagemaker as sage
from sagemaker import get_execution_role
import boto3

In [ ]:
role = get_execution_role()

sagemaker_session = sage.Session()
bucket = sagemaker_session.default_bucket()
runtime = boto3.client("runtime.sagemaker")
sm_client = boto3.client("sagemaker")

bucket

## 2. Create an endpoint and perform real-time inference

If you want to understand how real-time inference with Amazon SageMaker works, see [Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-hosting.html).

In [ ]:
endpoint_name = "m-optimus"
content_type = "application/json"
real_time_inference_instance_type = "ml.g5.xlarge"
batch_transform_inference_instance_type = "ml.g5.xlarge"

### A. Create an endpoint

In [ ]:
# Create a deployable model from the model package.
model = ModelPackage(
    role=role, model_package_arn=model_package_arn, sagemaker_session=sagemaker_session
)

In [ ]:
# Deploy the model.
predictor = model.deploy(
    1,
    real_time_inference_instance_type,
    endpoint_name=endpoint_name,
    inference_ami_version="al2-ami-sagemaker-inference-gpu-3-1"
)

Once endpoint has been created, you would be able to perform real-time inference.

### B. Load input payload (embedding mode)

In [ ]:
with open("data/input/real-time/m_embed_input.json") as f:
    embed_payload_str = f.read()

print(f"Payload size: {len(embed_payload_str.encode()) / 1024:.1f} KB")

The payload is a JSON object with the following fields:
- `image_data`: base64-encoded PNG of the tile
- `bulk_rna`: optional bulk RNA-seq vector (list of floats, or `null`)
- `slide_name`: identifier for the source image
- `x`, `y`, `width`, `height`: tile coordinates and size in pixels
- `tissue_ratio`: fraction of the tile covered by tissue
- `patch_idx`: index of this tile
- `resolution`: extraction resolution in MPP (0.5 for M-Optimus)
- `model_name`: model identifier used for server-side dispatch (`"m-optimus-stx"`)
- `mode`: `"embedding"` to request feature extraction

### C. Perform real-time inference (embedding mode)

In [ ]:
response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Accept="application/json",
    Body=embed_payload_str,
)

result = json.loads(response["Body"].read().decode("utf-8"))
features = result["output"]
assert len(features) == 1536, f"Unexpected embedding dimension: {len(features)}"
print(f"Embedding dimension: {len(features)}")
features[:5]

### D. Perform real-time inference (prediction mode)

In [ ]:
with open("data/input/real-time/m_predict_input.json") as f:
    predict_payload_str = f.read()

print(f"Payload size: {len(predict_payload_str.encode()) / 1024:.1f} KB")

response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Accept="application/json",
    Body=predict_payload_str,
)

result = json.loads(response["Body"].read().decode("utf-8"))
prediction = result["output"]
assert len(prediction) == 6002, f"Unexpected prediction dimension: {len(prediction)}"
print(f"Prediction output dimension: {len(prediction)}")
prediction[:5]

### E. Test tissue segmentation model

The tissue segmentation model (`tissue-seg`) operates on 512×512 tiles at 8.0 µm/px (MPP). It returns a flat binary mask of shape `(512 * 512,)` where each value is 0.0 (background) or 1.0 (tissue).

The input file `data/input/real-time/tissue_seg_input.json` contains a single record with:
- `image_data`: base64-encoded PNG of a 512×512 tile
- `width` / `height`: both 512
- `resolution`: 8.0
- `model_name`: `"tissue-seg"`
- `mode`: `"prediction"`

In [ ]:
with open("data/input/real-time/tissue_seg_input.json") as f:
    tissue_seg_payload_str = f.read()

print(f"Payload size: {len(tissue_seg_payload_str.encode()) / 1024:.1f} KB")

response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Accept="application/json",
    Body=tissue_seg_payload_str,
)

result = json.loads(response["Body"].read().decode("utf-8"))
mask = result["output"]
assert len(mask) == 512 * 512, f"Unexpected mask length: {len(mask)}"
tissue_pixels = sum(v > 0.5 for v in mask)
print(f"Mask length: {len(mask)}")
print(f"Tissue pixels: {tissue_pixels} / {512 * 512} ({100 * tissue_pixels / (512 * 512):.1f}%)")

### F. Delete the endpoint and model

Now that you have successfully performed real-time inference, you do not need the endpoint any more. You can terminate the endpoint to avoid being charged.

In [ ]:
model.sagemaker_session.delete_endpoint(endpoint_name)
model.sagemaker_session.delete_endpoint_config(endpoint_name)
model.delete_model()

## 3. Perform batch inference

In this section, you will perform batch inference using multiple input payloads together. The input files are already in JSONL format (one JSON record per line) as expected by the API — no additional conversion is needed. If you are not familiar with batch transform, and want to learn more, see these links:
1. [How it works](https://docs.aws.amazon.com/sagemaker/latest/dg/ex1-batch-transform.html)
2. [How to run a batch transform job](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-batch.html)

Create the model parameters

In [ ]:
batch_model_name = "m-optimus"
content_type = "application/json"
batch_transform_inference_instance_type = "ml.g5.xlarge"

Upload your batch data to S3. The JSONL files in `data/input/batch/` are already in the format expected by the API.

In [ ]:
# Upload both JSONL input files to S3.
transform_input = sagemaker_session.upload_data(
    "data/input/batch", key_prefix=batch_model_name
)
print("Transform input uploaded to " + transform_input)

Create output paths for the batch transform jobs

In [ ]:
timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
transform_output_embed = f"{transform_input}-embed-output-{timestamp}"
transform_output_predict = f"{transform_input}-predict-output-{timestamp}"
print("Embedding output path:", transform_output_embed)
print("Prediction output path:", transform_output_predict)

Create the SageMaker model based on the model package ARN

In [ ]:
print(f"Creating Model: {batch_model_name}...")
create_model_response = sm_client.create_model(
    ModelName=batch_model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "ModelPackageName": model_package_arn
    },
    EnableNetworkIsolation=True
)

### Embedding batch transform job

Runs the embedding mode across all records in `m_embed_inputs.jsonl`.

In [ ]:
embed_job_name = f"m-optimus-embed-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}"

print(f"Starting embedding transform job: {embed_job_name}...")
response = sm_client.create_transform_job(
    TransformJobName=embed_job_name,
    ModelName=batch_model_name,
    MaxConcurrentTransforms=1,
    MaxPayloadInMB=6,
    BatchStrategy="SingleRecord",
    TransformInput={
        "DataSource": {
            "S3DataSource": {
                "S3DataType": "S3Prefix",
                "S3Uri": transform_input + "/m_embed_inputs.jsonl"
            }
        },
        "ContentType": content_type,
        "SplitType": "Line",
        "CompressionType": "None"
    },
    TransformOutput={
        "S3OutputPath": transform_output_embed,
        "AssembleWith": "Line",
        "Accept": "application/json"
    },
    TransformResources={
        "TransformAmiVersion": "al2-ami-sagemaker-batch-gpu-535",
        "InstanceType": batch_transform_inference_instance_type,
        "InstanceCount": 1
    }
)

print(f"Transform Job ARN: {response['TransformJobArn']}")

In [ ]:
print("Waiting for embedding job to complete...")
start_time = time.time()
waiter = sm_client.get_waiter("transform_job_completed_or_stopped")
waiter.wait(TransformJobName=embed_job_name)
end_time = time.time()

duration_seconds = end_time - start_time
minutes = int(duration_seconds // 60)
seconds = int(duration_seconds % 60)

status = sm_client.describe_transform_job(TransformJobName=embed_job_name)
print(f"   Job finished with status: {status['TransformJobStatus']}")
print(f"   Total Wait Time: {minutes}m {seconds}s")

### Prediction batch transform job

Runs the prediction mode (image + bulk RNA-seq) across all records in `m_predict_inputs.jsonl`.

In [ ]:
predict_job_name = f"m-optimus-predict-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}"

print(f"Starting prediction transform job: {predict_job_name}...")
response = sm_client.create_transform_job(
    TransformJobName=predict_job_name,
    ModelName=batch_model_name,
    MaxConcurrentTransforms=1,
    MaxPayloadInMB=6,
    BatchStrategy="SingleRecord",
    TransformInput={
        "DataSource": {
            "S3DataSource": {
                "S3DataType": "S3Prefix",
                "S3Uri": transform_input + "/m_predict_inputs.jsonl"
            }
        },
        "ContentType": content_type,
        "SplitType": "Line",
        "CompressionType": "None"
    },
    TransformOutput={
        "S3OutputPath": transform_output_predict,
        "AssembleWith": "Line",
        "Accept": "application/json"
    },
    TransformResources={
        "TransformAmiVersion": "al2-ami-sagemaker-batch-gpu-535",
        "InstanceType": batch_transform_inference_instance_type,
        "InstanceCount": 1
    }
)

print(f"Transform Job ARN: {response['TransformJobArn']}")

In [ ]:
print("Waiting for prediction job to complete...")
start_time = time.time()
waiter = sm_client.get_waiter("transform_job_completed_or_stopped")
waiter.wait(TransformJobName=predict_job_name)
end_time = time.time()

duration_seconds = end_time - start_time
minutes = int(duration_seconds // 60)
seconds = int(duration_seconds % 60)

status = sm_client.describe_transform_job(TransformJobName=predict_job_name)
print(f"   Job finished with status: {status['TransformJobStatus']}")
print(f"   Total Wait Time: {minutes}m {seconds}s")

In [ ]:
# Output is available at the following paths
print("Embedding output:", transform_output_embed)
print("Prediction output:", transform_output_predict)

## 4. Clean-up

### A. Delete the model

In [ ]:
try:
    sm_client.delete_model(ModelName=batch_model_name)
    print(f"   Successfully deleted model: {batch_model_name}")
except Exception as cleanup_error:
    print(f"   Warning: Could not delete model. It may have already been deleted or never created.")
    print(f"   Error details: {cleanup_error}")

### B. Unsubscribe to the listing (optional)

If you would like to unsubscribe to the model package, follow these steps. Before you cancel the subscription, ensure that you do not have any [deployable model](https://console.aws.amazon.com/sagemaker/home#/models) created from the model package or using the algorithm. Note - You can find this information by looking at the container name associated with the model. 

**Steps to unsubscribe to product from AWS Marketplace**:
1. Navigate to __Machine Learning__ tab on [__Your Software subscriptions page__](https://aws.amazon.com/marketplace/ai/library?productType=ml&ref_=mlmp_gitdemo_indust)
2. Locate the listing that you want to cancel the subscription for, and then choose __Cancel Subscription__  to cancel the subscription.